In [1]:
import os
import sys

# using the local system spark which is installed via homebrew
os.environ["SPARK_HOME"] = "/opt/homebrew/opt/apache-spark/libexec"

sys.path.insert(
    0,
    "/opt/homebrew/opt/apache-spark/libexec/python"
)

sys.path.insert(
    0,
    "/opt/homebrew/opt/apache-spark/libexec/python/lib/py4j-0.10.9.9-src.zip"
)

In [2]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("MyApp") \
    .getOrCreate()

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/10 23:13:42 WARN Utils: Your hostname, Bikashs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.100.56 instead (on interface en0)
26/08/10 23:13:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/10 23:13:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


In [3]:
# function to read parquet files
def read_parquet_files(taxi, year):
    df = None
    for month in range(1, 13):
        month_str = str(month).zfill(2)
        file_path = f"data/pqt/{taxi}/{year}/{month_str}/{taxi}_tripdata_{year}_{month_str}.parquet"
        if os.path.exists(file_path):
            if df is None:
                df = spark.read.option("header", "true").parquet(file_path)
            else:
                df = df.union(spark.read.option("header", "true").parquet(file_path))
        else:
            print(f"File not found: {file_path}")
    
    if taxi == "green":
        df = df.withColumnRenamed("lpep_pickup_datetime", "pickup_datetime") \
                .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
    else:
        df = df.withColumnRenamed("tpep_pickup_datetime", "pickup_datetime") \
                .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
    return df

In [4]:
# read yellow taxi data for 2020
df_yellow = read_parquet_files("yellow", 2020)

# read green taxi data for 2020
df_green = read_parquet_files("green", 2020)

print(df_yellow.columns)
print(df_green.columns)

['VendorID', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee']
['VendorID', 'pickup_datetime', 'dropoff_datetime', 'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge', 'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge']


In [5]:
set(df_yellow.columns) & set(df_green.columns)

{'DOLocationID',
 'PULocationID',
 'RatecodeID',
 'VendorID',
 'congestion_surcharge',
 'dropoff_datetime',
 'extra',
 'fare_amount',
 'improvement_surcharge',
 'mta_tax',
 'passenger_count',
 'payment_type',
 'pickup_datetime',
 'store_and_fwd_flag',
 'tip_amount',
 'tolls_amount',
 'total_amount',
 'trip_distance'}

In [6]:
common_columns = []
yellow_columns = set(df_yellow.columns)
for col in df_green.columns:
    if col in yellow_columns:
        common_columns.append(col)

In [7]:
from pyspark.sql import functions as F

df_green_final = df_green.select(common_columns).withColumn('service_type', F.lit('green'))
df_yellow_final = df_yellow.select(common_columns).withColumn('service_type', F.lit('yellow'))

df_trips_data = df_green_final.union(df_yellow_final)

In [8]:
df_trips_data.show(5)

+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------------------+------------+------------+--------------------+------------+
|VendorID|    pickup_datetime|   dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|payment_type|congestion_surcharge|service_type|
+--------+-------------------+-------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------------------+------------+------------+--------------------+------------+
|       2|2019-12-18 15:52:30|2019-12-18 15:54:39|                 N|       1.0|         264|         264|            5.0|          0.0|        3.5|  0.5|    0.5|      0.01|         0.0|       

In [9]:
df_trips_data.groupBy("service_type").count().show()

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 1734176|
|      yellow|24649092|
+------------+--------+



## 🤯 Really huge dataset!!

In [10]:
## SQL with Spark
df_trips_data.createOrReplaceTempView("trips_data")

spark.sql("""
    SELECT 
        service_type, 
        COUNT(*) as trip_count
    FROM 
        trips_data
    GROUP BY 
        service_type
""").show()

+------------+----------+
|service_type|trip_count|
+------------+----------+
|       green|   1734176|
|      yellow|  24649092|
+------------+----------+



In [11]:
df_trips_data.columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'congestion_surcharge',
 'service_type']

In [12]:
spark.sql("""
    SELECT 
        PULocationID AS revenue_zone,
        date_trunc('month', pickup_datetime) AS revenue_month, 
        service_type, 
        ROUND(SUM(fare_amount), 2) AS revenue_monthly_fare,
        ROUND(SUM(extra), 2) AS revenue_monthly_extra,
        ROUND(SUM(mta_tax), 2) AS revenue_monthly_mta_tax,
        ROUND(SUM(tip_amount), 2) AS revenue_monthly_tip_amount,
        ROUND(SUM(tolls_amount), 2) AS revenue_monthly_tolls_amount,
        ROUND(SUM(improvement_surcharge), 2) AS revenue_monthly_improvement_surcharge,
        ROUND(SUM(total_amount), 2) AS revenue_monthly_total_amount,
        ROUND(SUM(congestion_surcharge), 2) AS revenue_monthly_congestion_surcharge,
        CEIL(AVG(passenger_count)) AS avg_monthly_passenger_count,
        ROUND(AVG(trip_distance), 2) AS avg_monthly_trip_distance
    FROM
        trips_data
    GROUP BY
        revenue_zone, revenue_month, service_type
    ORDER BY
        revenue_monthly_total_amount DESC
""").show()

+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip_amount|revenue_monthly_tolls_amount|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_monthly_passenger_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+--------------------------+----------------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|         132

## 💰 Insight: Yellow Taxi generates significantly higher revenue than Green Taxi.